# 05 - Putting It All Together

**Goal:** See how DVC + MLflow work together to give you full data-to-model traceability.

---

## The Full Picture

```
DVC                              MLflow
├── Data versioning              ├── Experiment tracking
├── Pipeline definition          ├── Metric comparison
├── Reproducibility              ├── Model registry
└── Data lineage                 └── Model deployment

Together:
  data (DVC) → training (DVC pipeline + MLflow tracking) → model (MLflow registry)
  
  "Which data trained the production model?" → answerable.
```

## Scenario: Two Dataset Versions, Two Models

We'll walk through a realistic scenario:

1. Version dataset v1 with DVC
2. Train model, track with MLflow → model v1
3. Get more data → dataset v2
4. Retrain → model v2
5. Compare v1 vs v2
6. Promote the best to Production

In [ ]:
import os
import shutil
import json
import numpy as np
import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch
import yaml

# Setup
DEMO_DIR = '/tmp/full-mlops-demo'
if os.path.exists(DEMO_DIR):
    shutil.rmtree(DEMO_DIR)
os.makedirs(DEMO_DIR)
os.chdir(DEMO_DIR)

# Initialize git + DVC
!git init
!git branch -m main
!dvc init

# DVC remote (local for demo, Azure in production)
!mkdir -p /tmp/full-mlops-storage
!dvc remote add -d local /tmp/full-mlops-storage

# MLflow tracking
MLFLOW_DIR = '/tmp/full-mlops-mlflow'
if os.path.exists(MLFLOW_DIR):
    shutil.rmtree(MLFLOW_DIR)
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
mlflow.set_experiment("spine-semantic-segmentation")

!git add .
!git commit -m "Initialize project with DVC + MLflow"

print("Ready.")

In [ ]:
# Reusable model (same as earlier notebooks)
class MiniUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4, features=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, features, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(features, features * 2, 3, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(features * 2, features, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(features, out_channels, 1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def compute_dice(preds, labels, num_classes):
    """Compute mean Dice score."""
    dice_scores = []
    for c in range(num_classes):
        pred_c = (preds == c).float()
        label_c = (labels == c).float()
        intersection = (pred_c * label_c).sum()
        dice = (2 * intersection / (pred_c.sum() + label_c.sum() + 1e-8)).item()
        dice_scores.append(dice)
    return np.mean(dice_scores), dice_scores

## Step 1: Dataset v1 (20 patients)

In [ ]:
# Create dataset v1
np.random.seed(42)
os.makedirs("data/spine", exist_ok=True)

n_samples = 20
images = np.random.randn(n_samples, 1, 32, 32).astype(np.float32)
labels = np.random.randint(0, 4, size=(n_samples, 32, 32)).astype(np.int64)

np.save("data/spine/images.npy", images)
np.save("data/spine/labels.npy", labels)

with open("data/spine/metadata.json", "w") as f:
    json.dump({"version": "v1", "num_patients": n_samples}, f)

# Track with DVC
!dvc add data/spine
!git add data/spine.dvc data/.gitignore
!git commit -m "Add spine dataset v1 (20 patients)"
!dvc push

print(f"Dataset v1: {n_samples} patients")

## Step 2: Train Model v1 + Track with MLflow

In [ ]:
def train_and_track(dataset_version, model_features, lr, num_epochs, run_name):
    """Full training with DVC data + MLflow tracking."""
    
    # Load DVC-tracked data
    images = torch.from_numpy(np.load("data/spine/images.npy"))
    labels = torch.from_numpy(np.load("data/spine/labels.npy"))
    
    # Split 80/20
    n_train = int(len(images) * 0.8)
    train_x, val_x = images[:n_train], images[n_train:]
    train_y, val_y = labels[:n_train], labels[n_train:]
    
    with mlflow.start_run(run_name=run_name) as run:
        # Log everything
        mlflow.log_param("dataset_version", dataset_version)
        mlflow.log_param("num_patients", len(images))
        mlflow.log_param("n_train", n_train)
        mlflow.log_param("n_val", len(images) - n_train)
        mlflow.log_param("features", model_features)
        mlflow.log_param("lr", lr)
        mlflow.log_param("num_epochs", num_epochs)
        
        # Get DVC data hash for traceability
        with open("data/spine.dvc") as f:
            dvc_info = yaml.safe_load(f)
        data_hash = dvc_info["outs"][0]["md5"]
        mlflow.log_param("dvc_data_hash", data_hash)
        mlflow.set_tag("dataset_version", dataset_version)
        
        # Train
        model = MiniUNet(features=model_features)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        for epoch in range(num_epochs):
            output = model(train_x)
            loss = criterion(output, train_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            mlflow.log_metric("train_loss", loss.item(), step=epoch)
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            val_output = model(val_x)
            val_preds = val_output.argmax(dim=1)
            mean_dice, per_class = compute_dice(val_preds, val_y, num_classes=4)
        
        mlflow.log_metric("val_dice_mean", mean_dice)
        for i, d in enumerate(per_class):
            mlflow.log_metric(f"val_dice_class_{i}", d)
        
        # Log model
        mlflow.pytorch.log_model(model, "model")
        
        # Log the DVC file as artifact (links to exact data version)
        mlflow.log_artifact("data/spine.dvc")
        
        print(f"Run '{run_name}' | dice={mean_dice:.4f} | data={dataset_version} | hash={data_hash[:8]}")
        return run.info.run_id, mean_dice


# Train model v1 on dataset v1
run_v1_id, dice_v1 = train_and_track(
    dataset_version="v1",
    model_features=16,
    lr=0.001,
    num_epochs=30,
    run_name="model-v1-data-v1"
)

## Step 3: Get More Data → Dataset v2

In [ ]:
# Add 30 more patients (total: 50)
np.random.seed(123)
more_images = np.random.randn(30, 1, 32, 32).astype(np.float32)
more_labels = np.random.randint(0, 4, size=(30, 32, 32)).astype(np.int64)

old_images = np.load("data/spine/images.npy")
old_labels = np.load("data/spine/labels.npy")

all_images = np.concatenate([old_images, more_images])
all_labels = np.concatenate([old_labels, more_labels])

np.save("data/spine/images.npy", all_images)
np.save("data/spine/labels.npy", all_labels)

with open("data/spine/metadata.json", "w") as f:
    json.dump({"version": "v2", "num_patients": 50}, f)

# Update DVC tracking
!dvc add data/spine
!git add data/spine.dvc
!git commit -m "Update spine dataset to v2 (50 patients)"
!dvc push

print(f"Dataset v2: {len(all_images)} patients")

## Step 4: Train Model v2 on Dataset v2

In [ ]:
# Train model v2 on dataset v2
run_v2_id, dice_v2 = train_and_track(
    dataset_version="v2",
    model_features=16,
    lr=0.001,
    num_epochs=30,
    run_name="model-v2-data-v2"
)

## Step 5: Compare v1 vs v2

In [ ]:
import pandas as pd

# Query all runs
experiment = mlflow.get_experiment_by_name("spine-semantic-segmentation")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.val_dice_mean DESC"],
)

comparison = runs[[
    "tags.mlflow.runName",
    "params.dataset_version",
    "params.num_patients",
    "params.dvc_data_hash",
    "metrics.val_dice_mean",
]].copy()
comparison.columns = ["run_name", "data_version", "patients", "data_hash", "dice"]
comparison["data_hash"] = comparison["data_hash"].str[:12]

print("Experiment Comparison:")
print(comparison.to_string(index=False))

## Step 6: Register and Promote Best Model

In [ ]:
model_name = "spine-semantic-model"

# Register both
reg_v1 = mlflow.register_model(f"runs:/{run_v1_id}/model", model_name)
reg_v2 = mlflow.register_model(f"runs:/{run_v2_id}/model", model_name)

print(f"Registered v1 (dice={dice_v1:.4f}) and v2 (dice={dice_v2:.4f})")

In [ ]:
# Promote the best to Production
client = mlflow.tracking.MlflowClient()

best_version = "1" if dice_v1 > dice_v2 else "2"
other_version = "2" if best_version == "1" else "1"

client.transition_model_version_stage(model_name, best_version, "Production")
client.transition_model_version_stage(model_name, other_version, "Archived")

print(f"\nModel registry:")
for mv in client.search_model_versions(f"name='{model_name}'"):
    run = client.get_run(mv.run_id)
    dice = run.data.metrics.get("val_dice_mean", "N/A")
    data_ver = run.data.params.get("dataset_version", "N/A")
    print(f"  Version {mv.version} | dice={dice:.4f} | data={data_ver} | stage={mv.current_stage}")

## FDA Traceability: Answering the Hard Questions

With DVC + MLflow, every question about your deployed model is answerable:

In [ ]:
# Question 1: "Which model is in production?"
prod_versions = [mv for mv in client.search_model_versions(f"name='{model_name}'")
                 if mv.current_stage == "Production"]
prod_model = prod_versions[0]
print(f"Q: Which model is in production?")
print(f"A: {model_name} version {prod_model.version}")
print()

In [ ]:
# Question 2: "What was its validation performance?"
run = client.get_run(prod_model.run_id)
print(f"Q: What was its validation performance?")
print(f"A: Mean Dice = {run.data.metrics['val_dice_mean']:.4f}")
for key, val in sorted(run.data.metrics.items()):
    if key.startswith("val_dice_class"):
        print(f"   {key} = {val:.4f}")
print()

In [ ]:
# Question 3: "Which data trained this model?"
print(f"Q: Which data trained this model?")
print(f"A: Dataset version: {run.data.params['dataset_version']}")
print(f"   Patients: {run.data.params['num_patients']}")
print(f"   DVC hash: {run.data.params['dvc_data_hash']}")
print(f"   (Run 'git log' to find the commit with this .dvc hash)")
print()

In [ ]:
# Question 4: "What hyperparameters were used?"
print(f"Q: What hyperparameters were used?")
for key, val in sorted(run.data.params.items()):
    if key not in ('dvc_data_hash',):
        print(f"   {key} = {val}")

## DVC + MLflow vs Azure ML Studio

| Feature | DVC + MLflow | Azure ML Studio |
|---------|-------------|------------------|
| **Data versioning** | DVC (`.dvc` files in Git) | Data Assets (built-in) |
| **Experiment tracking** | MLflow (params, metrics, artifacts) | Built-in (same concepts) |
| **Model registry** | MLflow Registry | Built-in Registry |
| **Pipelines** | DVC pipelines (`dvc.yaml`) | Azure ML Pipelines (Python SDK) |
| **Compute** | Your own machines / Docker | Azure Compute (pay per use) |
| **UI** | MLflow UI (self-hosted) | Azure Portal (managed) |
| **Cost** | Free (open source) | Azure subscription |
| **Vendor lock-in** | None | Azure |
| **Setup effort** | Moderate (you configure everything) | Low (managed service) |
| **Flexibility** | Full control | Azure's way |

**For your startup:**
- You have Azure credits → try both
- DVC + MLflow = portable, free, full control
- Azure ML = nice UI, less setup, managed compute
- Many teams use a hybrid: DVC for data + Azure ML for compute + MLflow for tracking

## Moving to Azure (When You're Ready)

The transition is mostly config changes:

**DVC → Azure Blob Storage:**
```bash
# Replace local remote with Azure
pip install dvc-azure
dvc remote modify local url azure://maia-data-container/dvc-storage
dvc remote modify local account_name maiasaaccount
# Everything else stays the same: dvc push, dvc pull, etc.
```

**MLflow → Remote tracking server:**
```bash
# Instead of file:// tracking, point to a server
mlflow.set_tracking_uri("http://mlflow-server.internal:5000")
# Or Azure ML workspace:
mlflow.set_tracking_uri(azureml://...)
# All mlflow.log_* calls stay exactly the same
```

**Your code doesn't change.** Only the config does.

## Recommendations for Your ML Team

**Immediate (this week):**
1. Add DVC to the training repo
2. Track data and splits with `dvc add`
3. Commit `.dvc` files with every training run

**Short-term (this month):**
4. Add MLflow logging to `training_semantic.py` and `training_instance.py`
5. Set up a shared MLflow server (can be a simple Docker container)
6. Define a `dvc.yaml` pipeline for the training workflow

**Medium-term (this quarter):**
7. Create a `maia-data-registry` repo
8. Set up Azure Blob as DVC remote
9. Set up MLflow Model Registry with stage gates
10. Document the workflow for FDA audits

**The key is to start small.** Even just `dvc add` on your data folders gives you traceability that doesn't exist today.

## Summary

| Tool | Tracks | Why |
|------|--------|-----|
| **Git** | Code + configs | Source of truth for logic |
| **DVC** | Data + pipelines | Source of truth for data |
| **MLflow** | Experiments + models | Source of truth for results |

Together they answer:
- **What code?** → Git commit
- **What data?** → DVC hash in `.dvc` file
- **What config?** → MLflow parameters (+ Git)
- **What result?** → MLflow metrics
- **Which model?** → MLflow Model Registry

**This is the data lineage that FDA 510(k) requires.** From deployed model → training run → exact data version → exact code version. Every link is traceable.

**Phase 4 complete.** You now understand the MLOps tools. Phase 5 will build a mini version of the full spine pipeline with all of this integrated.